# Model A — Segmentation Model

## 현재 단계

Segmentation GT Rasterization은 전체 **8,788장**에서 검증 완료되었다.

현재 Segmentation Head의 Progressive Decoder 구조와 Dummy Forward 검증까지 완료했으며,
다음 단계는 검증된 Head를 `src/segmentation/head.py`로 모듈화하는 것이다.

### 현재 작업 순서

1. Shared Encoder → Segmentation Head Input Contract 확인 ✅
2. Segmentation Head 설계 ✅
3. Dummy Feature 기반 Head Forward 검증 ✅
4. Head 구조 정리 및 `src/segmentation/head.py` 모듈화 ← **현재**
5. Loss 설계
6. Dataset / DataLoader 연결
7. Prediction ↔ GT 연결 검증
8. Training / Validation
9. Prediction QC

> Dataset 관련 설계는 폐기하지 않는다.  
> 다만 Model A의 공통 Data Pipeline과 연결될 가능성이 높으므로,
> Segmentation Head와 Loss의 윤곽을 먼저 확정한 뒤 구현한다.

# 1. Shared Encoder → Segmentation Head Input Contract

## 역할 구분

Shared Encoder 구현 및 최종 Multi-task 통합은 **이주한 담당**이다.

Segmentation에서는 별도의 Encoder를 구현하지 않는다.

태건 Segmentation Head는 주한 Shared Encoder가 반환하는
multi-scale feature를 입력으로 사용한다.

## Shared Encoder

Backbone:

- `ConvNeXt-Tiny`
- ImageNet pretrained
- `features_only=True`
- `out_indices=(0, 1, 2, 3)`

실제 Model A Encoder 입력:

- `[B, 3, 448, 768]`

실제 Forward 결과:

- `c1`: `[B, 96, 112, 192]`, stride 4
- `c2`: `[B, 192, 56, 96]`, stride 8
- `c3`: `[B, 384, 28, 48]`, stride 16
- `c4`: `[B, 768, 14, 24]`, stride 32

Encoder 반환 형식:

```python
{
    "c1": c1,
    "c2": c2,
    "c3": c3,
    "c4": c4,
}
```

## Segmentation Head Output Contract

Segmentation Head의 최종 출력:

- `[B, 3, 448, 768]` logits

Channel 의미:

- channel 0: `traffic_lane`
- channel 1: `stop_line`
- channel 2: `crosswalk`

세 channel은 서로 배타적인 class ID가 아니다.

각 channel은 독립적인 binary logit이며,
세 channel이 모두 negative인 pixel은 implicit background로 해석한다.


# 2. Segmentation Head Design

## 목적

Shared Encoder가 반환하는 `C1 ~ C4` multi-scale feature를 이용하여
traffic_lane / stop_line / crosswalk를 예측하는 Segmentation Head를 설계한다.

아직 `head.py` 구현을 바로 확정하지 않는다.

진행 순서:

1. 사용할 Encoder feature stage 결정
2. Feature fusion / decoder 구조 비교
3. Decoder channel 설정
4. Upsampling 방식 결정
5. Dummy feature로 Forward 검증
6. 구조가 검증된 뒤 `src/segmentation/head.py`로 모듈화

## Task 특성

### traffic_lane
- 얇고 긴 구조
- 정확한 위치 정보가 중요
- 동시에 전체 lane-flow / 도로 구조 context도 필요

### stop_line
- 얇은 선 구조
- 위치 정확도가 특히 중요

### crosswalk
- polygon 기반의 넓은 region
- local boundary와 scene-level context가 모두 중요

## Multi-scale feature를 검토하는 이유

Encoder의 얕은 feature와 깊은 feature는 서로 다른 수준의 정보를 가진다.

- `C1 / C2`
  - 상대적으로 높은 spatial resolution
  - 얇은 lane / stop_line의 위치와 경계 복원에 유리할 가능성

- `C3 / C4`
  - 더 깊은 semantic / context 정보
  - 도로 구조, lane flow, crosswalk region 판단에 유리할 가능성

`C1 ~ C4` 전체 사용이 항상 최적이라고 미리 확정하지 않는다.

Baseline 구조를 만든 뒤 stage 조합은 ablation을 통해 비교할 수 있도록 설계한다.


## 2-1. 현재 결정해야 할 사항

다음 항목을 하나씩 결정한다.

1. 어떤 feature stage를 사용할 것인가?
   - `C1 ~ C4` 전체
   - 또는 일부 stage 조합

2. Decoder / Feature Fusion 구조
   - Progressive Multi-scale Decoder를 첫 baseline으로 선택

3. 각 stage channel projection 크기
   - 초기 baseline 후보: 256

4. Feature fusion 방식
   - 초기 baseline 후보: concatenation

5. Upsampling 방식
   - 초기 baseline 후보: bilinear interpolation

6. 최종 출력
   - `[B, 3, 448, 768]` logits

위 값들은 최종 고정값이 아니며,
Forward 검증 및 학습 결과에 따라 조정한다.


## 2-2. Decoder 구조 후보

비교 후보는 다음 두 가지로 정리하였다.

### Candidate A — FPN-style Decoder

- C1~C4의 channel을 projection하여 통일
- C4부터 top-down upsampling
- 각 scale의 lateral feature와 fusion
- multi-scale feature를 이용해 최종 segmentation logits 생성

### Candidate B — Progressive Decoder

- C4에서 시작
- C3 → C2 → C1 순서로 feature를 단계적으로 결합
- 각 단계에서 upsampling + feature fusion + decoder block 수행
- 최종적으로 GT 해상도까지 복원

현재 첫 baseline은 **Progressive Multi-scale Decoder**로 진행한다.


## 2-3. Baseline Decoder 결정

첫 Segmentation Head baseline은
`C4 → C3 → C2 → C1` 순서로 feature를 단계적으로 복원하는
**Progressive Multi-scale Decoder**를 사용한다.

선택 이유:

- Shared Encoder feature가 stride `32 → 16 → 8 → 4`의 계층적 구조를 가진다.
- traffic_lane / stop_line은 세밀한 spatial information이 중요하다.
- crosswalk는 deeper feature의 semantic/context information도 필요하다.
- deep feature와 shallow feature를 단계적으로 결합하기에 적합하다.

초기 baseline 후보:

- decoder channel: `256`
- upsampling: `bilinear interpolation`
- fusion: `concatenation`
- final output: `[B, 3, 448, 768]`

본 구조를 최종 구조로 고정하지 않으며,
baseline 학습 결과에 따라 stage 조합 및 decoder 구조를 비교한다.


## 2-4. Progressive Decoder Shape Test — C4 → C3

먼저 Progressive Decoder 전체를 한 번에 구현하지 않는다.

첫 번째 단계인 `C4 → C3` fusion만 실험하여
Tensor shape 흐름이 의도대로 동작하는지 확인한다.

현재 목표:

```text
C4 [B, 768, 14, 24]
 ↓ 1×1 Conv
[B, 256, 14, 24]
 ↓ bilinear upsample
[B, 256, 28, 48]

C3 [B, 384, 28, 48]
 ↓ 1×1 Conv
[B, 256, 28, 48]

두 feature를 channel 방향으로 concat
 ↓
[B, 512, 28, 48]

3×3 Conv
 ↓
D3 [B, 256, 28, 48]
```

`D3`는 모델 이름이 아니라,
C4와 C3 정보를 합쳐 Decoder가 만든 **중간 feature map**을 뜻하는 임시 이름이다.


In [22]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())


PyTorch: 2.13.0+cu126
CUDA build: 12.6
CUDA available: True


In [23]:
# ============================================================
# Progressive Decoder Dummy Feature
# - Shared Encoder 실제 output shape 기준
# ============================================================

BATCH_SIZE = 2

c3 = torch.randn(
    BATCH_SIZE,
    384,
    28,
    48,
)

c4 = torch.randn(
    BATCH_SIZE,
    768,
    14,
    24,
)

print("C3 shape:", c3.shape)
print("C4 shape:", c4.shape)


C3 shape: torch.Size([2, 384, 28, 48])
C4 shape: torch.Size([2, 768, 14, 24])


In [24]:
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# C4 → C3 Fusion Shape Test
# ============================================================

DECODER_CHANNELS = 256

# C4 channel projection: 768 → 256
c4_projection = nn.Conv2d(
    in_channels=768,
    out_channels=DECODER_CHANNELS,
    kernel_size=1,
)

# C3 channel projection: 384 → 256
c3_projection = nn.Conv2d(
    in_channels=384,
    out_channels=DECODER_CHANNELS,
    kernel_size=1,
)

c4_projected = c4_projection(c4)
c3_projected = c3_projection(c3)

print("C4 projected:", c4_projected.shape)
print("C3 projected:", c3_projected.shape)

# C4를 C3 spatial size로 upsample
c4_upsampled = F.interpolate(
    c4_projected,
    size=c3_projected.shape[-2:],
    mode="bilinear",
    align_corners=False,
)

print("C4 upsampled:", c4_upsampled.shape)

# Channel 방향 concat
fused_c43 = torch.cat(
    [c4_upsampled, c3_projected],
    dim=1,
)

print("C4 + C3 fused:", fused_c43.shape)


C4 projected: torch.Size([2, 256, 14, 24])
C3 projected: torch.Size([2, 256, 28, 48])
C4 upsampled: torch.Size([2, 256, 28, 48])
C4 + C3 fused: torch.Size([2, 512, 28, 48])


In [25]:
# ============================================================
# C4 + C3 Fusion → D3
# ============================================================

c43_fusion_conv = nn.Conv2d(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
    kernel_size=3,
    padding=1,
)

d3 = c43_fusion_conv(fused_c43)

print("Fused C4 + C3:", fused_c43.shape)
print("D3 shape:", d3.shape)


Fused C4 + C3: torch.Size([2, 512, 28, 48])
D3 shape: torch.Size([2, 256, 28, 48])


### 2-4-1. C4 → C3 단계 기대 결과

정상이라면 다음 shape이 나온다.

```text
C4 projected : [B, 256, 14, 24]
C3 projected : [B, 256, 28, 48]
C4 upsampled : [B, 256, 28, 48]
C4 + C3 fused: [B, 512, 28, 48]
D3            : [B, 256, 28, 48]
```

여기까지 확인되면 Progressive Decoder의 첫 번째 fusion stage는
shape 관점에서 정상 동작한 것이다.


## 2-5. Progressive Decoder Shape Test — D3 → C2 → D2

`C4 + C3 → D3`가 정상 동작한 뒤,
D3를 C2 해상도로 확대하고 C2를 256 channel로 projection하여 fusion한다.

정상 기대 흐름:

```text
D3 [B, 256, 28, 48]
 ↓ upsample
[B, 256, 56, 96]

C2 [B, 192, 56, 96]
 ↓ 1×1 Conv
[B, 256, 56, 96]

concat
 ↓
[B, 512, 56, 96]

3×3 Conv
 ↓
D2 [B, 256, 56, 96]
```

In [26]:
# ============================================================
# C2 Dummy Feature
# - Shared Encoder 실제 output shape 기준
# ============================================================

c2 = torch.randn(
    BATCH_SIZE,
    192,
    56,
    96,
)

print("C2 shape:", c2.shape)

C2 shape: torch.Size([2, 192, 56, 96])


In [27]:
# ============================================================
# D3 + C2 Fusion → D2
# ============================================================

c2_projection = nn.Conv2d(
    in_channels=192,
    out_channels=DECODER_CHANNELS,
    kernel_size=1,
)

c2_projected = c2_projection(c2)

d3_upsampled = F.interpolate(
    d3,
    size=c2_projected.shape[-2:],
    mode="bilinear",
    align_corners=False,
)

fused_d3c2 = torch.cat(
    [d3_upsampled, c2_projected],
    dim=1,
)

d3c2_fusion_conv = nn.Conv2d(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
    kernel_size=3,
    padding=1,
)

d2 = d3c2_fusion_conv(fused_d3c2)

print("C2 projected :", c2_projected.shape)
print("D3 upsampled :", d3_upsampled.shape)
print("D3 + C2 fused:", fused_d3c2.shape)
print("D2            :", d2.shape)

C2 projected : torch.Size([2, 256, 56, 96])
D3 upsampled : torch.Size([2, 256, 56, 96])
D3 + C2 fused: torch.Size([2, 512, 56, 96])
D2            : torch.Size([2, 256, 56, 96])


## 2-6. Progressive Decoder Shape Test — D2 → C1 → D1

D2를 C1 해상도로 확대하고 C1을 256 channel로 projection한 뒤 fusion한다.

정상 기대 흐름:

```text
D2 [B, 256, 56, 96]
 ↓ upsample
[B, 256, 112, 192]

C1 [B, 96, 112, 192]
 ↓ 1×1 Conv
[B, 256, 112, 192]

concat
 ↓
[B, 512, 112, 192]

3×3 Conv
 ↓
D1 [B, 256, 112, 192]
```

In [28]:
# ============================================================
# C1 Dummy Feature
# - Shared Encoder 실제 output shape 기준
# ============================================================

c1 = torch.randn(
    BATCH_SIZE,
    96,
    112,
    192,
)

print("C1 shape:", c1.shape)

C1 shape: torch.Size([2, 96, 112, 192])


In [29]:
# ============================================================
# D2 + C1 Fusion → D1
# ============================================================

c1_projection = nn.Conv2d(
    in_channels=96,
    out_channels=DECODER_CHANNELS,
    kernel_size=1,
)

c1_projected = c1_projection(c1)

d2_upsampled = F.interpolate(
    d2,
    size=c1_projected.shape[-2:],
    mode="bilinear",
    align_corners=False,
)

fused_d2c1 = torch.cat(
    [d2_upsampled, c1_projected],
    dim=1,
)

d2c1_fusion_conv = nn.Conv2d(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
    kernel_size=3,
    padding=1,
)

d1 = d2c1_fusion_conv(fused_d2c1)

print("C1 projected :", c1_projected.shape)
print("D2 upsampled :", d2_upsampled.shape)
print("D2 + C1 fused:", fused_d2c1.shape)
print("D1            :", d1.shape)

C1 projected : torch.Size([2, 256, 112, 192])
D2 upsampled : torch.Size([2, 256, 112, 192])
D2 + C1 fused: torch.Size([2, 512, 112, 192])
D1            : torch.Size([2, 256, 112, 192])


## 2-7. D1 → Final Segmentation Logits

D1의 256 channel을 3개 독립 binary logit channel로 변환하고,
최종 GT 해상도까지 upsample한다.

```text
D1 [B, 256, 112, 192]
 ↓ 1×1 Conv
[B, 3, 112, 192]
 ↓ bilinear upsample
[B, 3, 448, 768]
```

In [30]:
# ============================================================
# D1 → Final Segmentation Logits
# ============================================================

seg_classifier = nn.Conv2d(
    in_channels=DECODER_CHANNELS,
    out_channels=3,
    kernel_size=1,
)

seg_logits_lowres = seg_classifier(d1)

seg_logits = F.interpolate(
    seg_logits_lowres,
    size=(448, 768),
    mode="bilinear",
    align_corners=False,
)

print("Low-res logits:", seg_logits_lowres.shape)
print("Final logits  :", seg_logits.shape)

Low-res logits: torch.Size([2, 3, 112, 192])
Final logits  : torch.Size([2, 3, 448, 768])


## 2-8. Decoder Block Baseline

초기 shape 실험에서는 fusion 뒤 `3×3 Conv`만 사용했다.
현재 학습용 baseline에서는 아래 Decoder Block을 사용한다.

```text
Concat
 ↓
3×3 Conv
 ↓
GroupNorm
 ↓
GELU
 ↓
Decoder Feature
```

- Conv: fusion된 feature를 학습하여 섞고 channel을 정리
- GroupNorm: batch size에 덜 의존하도록 feature 분포를 정돈
- GELU: 비선형성 추가

현재 baseline은 `decoder_channels=256`, `num_groups=32`이다.

In [31]:
# ============================================================
# Decoder Block
# Conv → GroupNorm → GELU
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        num_groups=32,
    ):
        super().__init__()

        if out_channels % num_groups != 0:
            raise ValueError(
                f"out_channels({out_channels}) must be divisible "
                f"by num_groups({num_groups})."
            )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(
                num_groups=num_groups,
                num_channels=out_channels,
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)

In [32]:
# ============================================================
# Decoder Block Dummy Test
# ============================================================

decoder_block_test = DecoderBlock(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
)

d3_block_test = decoder_block_test(fused_c43)

print("Input :", fused_c43.shape)
print("Output:", d3_block_test.shape)
print("NaN   :", torch.isnan(d3_block_test).any().item())
print("Inf   :", torch.isinf(d3_block_test).any().item())

Input : torch.Size([2, 512, 28, 48])
Output: torch.Size([2, 256, 28, 48])
NaN   : False
Inf   : False


## 2-9. Final SegmentationHead Candidate

아래 클래스는 지금까지 검증한 Progressive Decoder baseline을
하나의 재사용 가능한 `nn.Module`로 정리한 현재 후보이다.

설계 원칙:

- Encoder channel 수는 `encoder_channels` 인자로 받는다.
- Decoder channel 수는 `decoder_channels` 인자로 받는다.
- 최종 출력 크기는 Head 내부에 `448×768`로 hardcoding하지 않고
  `forward(..., output_size=...)`에서 외부로부터 받는다.
- Decoder fusion block은 `Conv → GroupNorm → GELU`를 사용한다.

In [33]:
# ============================================================
# Progressive Segmentation Head
# ============================================================

class SegmentationHead(nn.Module):

    def __init__(
        self,
        encoder_channels=(96, 192, 384, 768),
        decoder_channels=256,
        num_classes=3,
        num_groups=32,
    ):
        super().__init__()

        if len(encoder_channels) != 4:
            raise ValueError(
                "encoder_channels must contain exactly 4 values "
                "for c1, c2, c3, c4."
            )

        c1_channels, c2_channels, c3_channels, c4_channels = (
            encoder_channels
        )

        self.c4_projection = nn.Conv2d(
            c4_channels,
            decoder_channels,
            kernel_size=1,
        )
        self.c3_projection = nn.Conv2d(
            c3_channels,
            decoder_channels,
            kernel_size=1,
        )
        self.c2_projection = nn.Conv2d(
            c2_channels,
            decoder_channels,
            kernel_size=1,
        )
        self.c1_projection = nn.Conv2d(
            c1_channels,
            decoder_channels,
            kernel_size=1,
        )

        fusion_channels = decoder_channels * 2

        self.c43_fusion = DecoderBlock(
            in_channels=fusion_channels,
            out_channels=decoder_channels,
            num_groups=num_groups,
        )
        self.d3c2_fusion = DecoderBlock(
            in_channels=fusion_channels,
            out_channels=decoder_channels,
            num_groups=num_groups,
        )
        self.d2c1_fusion = DecoderBlock(
            in_channels=fusion_channels,
            out_channels=decoder_channels,
            num_groups=num_groups,
        )

        self.classifier = nn.Conv2d(
            decoder_channels,
            num_classes,
            kernel_size=1,
        )

    def forward(
        self,
        features,
        output_size,
    ):
        required_keys = ("c1", "c2", "c3", "c4")
        missing_keys = [
            key for key in required_keys
            if key not in features
        ]

        if missing_keys:
            raise KeyError(
                f"Missing encoder feature keys: {missing_keys}"
            )

        c1 = features["c1"]
        c2 = features["c2"]
        c3 = features["c3"]
        c4 = features["c4"]

        # C4 + C3 → D3
        c4 = self.c4_projection(c4)
        c3 = self.c3_projection(c3)
        c4 = F.interpolate(
            c4,
            size=c3.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        d3 = torch.cat([c4, c3], dim=1)
        d3 = self.c43_fusion(d3)

        # D3 + C2 → D2
        c2 = self.c2_projection(c2)
        d3 = F.interpolate(
            d3,
            size=c2.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        d2 = torch.cat([d3, c2], dim=1)
        d2 = self.d3c2_fusion(d2)

        # D2 + C1 → D1
        c1 = self.c1_projection(c1)
        d2 = F.interpolate(
            d2,
            size=c1.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        d1 = torch.cat([d2, c1], dim=1)
        d1 = self.d2c1_fusion(d1)

        # Final Segmentation Logits
        logits = self.classifier(d1)
        logits = F.interpolate(
            logits,
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        return logits

# 3. Final Head Dummy Forward Test

현재 Segmentation Head baseline 전체를 Dummy `C1 ~ C4` feature로 검증한다.

검증 목표:

- batch dimension 유지
- 최종 logits shape `[B, 3, 448, 768]`
- NaN 없음
- Inf 없음

In [34]:
# ============================================================
# Final SegmentationHead Dummy Forward Test
# ============================================================

BATCH_SIZE = 2

dummy_features = {
    "c1": torch.randn(BATCH_SIZE, 96, 112, 192),
    "c2": torch.randn(BATCH_SIZE, 192, 56, 96),
    "c3": torch.randn(BATCH_SIZE, 384, 28, 48),
    "c4": torch.randn(BATCH_SIZE, 768, 14, 24),
}

seg_head = SegmentationHead(
    encoder_channels=(96, 192, 384, 768),
    decoder_channels=256,
    num_classes=3,
    num_groups=32,
)

logits = seg_head(
    dummy_features,
    output_size=(448, 768),
)

print("Segmentation Head output:", logits.shape)
print("NaN:", torch.isnan(logits).any().item())
print("Inf:", torch.isinf(logits).any().item())

Segmentation Head output: torch.Size([2, 3, 448, 768])
NaN: False
Inf: False


정상 기대 결과:

```text
Segmentation Head output: torch.Size([2, 3, 448, 768])
NaN: False
Inf: False
```

이 결과가 나오면 현재 Progressive Segmentation Head baseline의
Dummy Forward 검증은 완료된 것으로 본다.

# 4. `src/segmentation/head.py` 모듈화 — 다음 단계

현재 다음 단계는 Notebook에서 검증한 아래 두 클래스를
`src/segmentation/head.py`로 옮기는 것이다.

- `DecoderBlock`
- `SegmentationHead`

모듈화 후에는 Notebook에서 `src.segmentation.head`로 import하여
동일한 Dummy Forward 검증을 다시 수행한다.

검증 항목:

- import 성공
- output shape `[B, 3, 448, 768]`
- NaN `False`
- Inf `False`

> 이 검증까지 통과한 시점을 **Segmentation Head baseline 완료 체크포인트**로 잡고
> GitHub commit / push를 진행한다.

# 5. Dataset / Data Pipeline 설계 — 보류

## 현재 상태

아래 Dataset 설계는 **삭제하지 않고 보존**한다.

다만 현재 태건의 우선 작업은 Segmentation Head이므로
Dataset 구현 / DataLoader 검증을 먼저 진행하지 않는다.

Model A 최종 구조는:

```text
RGB
 ↓
Shared Encoder
 ├─ Segmentation Head
 └─ Vector Head
```

이므로 Dataset / preprocessing은 최종적으로
Segmentation과 Vector가 공유하는 Model A 공통 Data Pipeline과 연결될 가능성이 높다.

따라서 아래 내용은 **설계 메모 / 향후 구현 contract**로 유지한다.


In [35]:
from pathlib import Path
import json


def find_project_root(start_path=None):
    current = Path(
        start_path if start_path is not None else Path.cwd()
    ).resolve()

    candidates = [current, *current.parents]

    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()

ANNOTATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "annotations"
    / "modelA_segmentation_annotations_v3.jsonl"
)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print()
print("ANNOTATION_PATH:")
print(ANNOTATION_PATH)
print()
print("annotation exists:", ANNOTATION_PATH.exists())


PROJECT_ROOT:
C:\Users\user\Desktop\Accident_AI

ANNOTATION_PATH:
C:\Users\user\Desktop\Accident_AI\data\annotations\modelA_segmentation_annotations_v3.jsonl

annotation exists: True


## 5-1. v3 JSONL Schema 확인

최종 Segmentation source annotation:

`data/annotations/modelA_segmentation_annotations_v3.jsonl`

Dataset 구현 단계에서 실제 schema를 다시 확인하기 위한 reference cell이다.


In [36]:
with ANNOTATION_PATH.open("r", encoding="utf-8") as f:
    first_line = f.readline()

first_annotation = json.loads(first_line)

print("Top-level keys:")
print(list(first_annotation.keys()))
print()
print("First annotation:")
print(json.dumps(first_annotation, ensure_ascii=False, indent=2))


Top-level keys:
['file_name', 'image_size', 'annotation_id', 'class', 'lane_type', 'category', 'data']

First annotation:
{
  "file_name": "13883084.jpg",
  "image_size": [
    1080,
    1920
  ],
  "annotation_id": 0,
  "class": "traffic_lane",
  "lane_type": "solid",
  "category": "polyline",
  "data": [
    {
      "x": 207,
      "y": 724
    },
    {
      "x": 206,
      "y": 723
    },
    {
      "x": 480,
      "y": 663
    },
    {
      "x": 832,
      "y": 584
    }
  ]
}


In [37]:
# ============================================================
# src/segmentation/head.py Import + Regression Test
# ============================================================

import sys
import torch


# ------------------------------------------------------------
# Project root를 Python import path에 추가
# ------------------------------------------------------------

project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:
    sys.path.insert(
        0,
        project_root_str,
    )


# ------------------------------------------------------------
# 실제 src의 SegmentationHead import
# - Notebook 안에서 정의한 클래스와 구분하기 위해 alias 사용
# ------------------------------------------------------------

from src.segmentation.head import (
    SegmentationHead as SegmentationHeadFromSrc,
)


# ------------------------------------------------------------
# Shared Encoder 실제 output shape 기준 Dummy Feature
# ------------------------------------------------------------

BATCH_SIZE = 2

dummy_features_src = {
    "c1": torch.randn(
        BATCH_SIZE, 96, 112, 192
    ),
    "c2": torch.randn(
        BATCH_SIZE, 192, 56, 96
    ),
    "c3": torch.randn(
        BATCH_SIZE, 384, 28, 48
    ),
    "c4": torch.randn(
        BATCH_SIZE, 768, 14, 24
    ),
}


# ------------------------------------------------------------
# src의 SegmentationHead 생성
# ------------------------------------------------------------

seg_head_src = SegmentationHeadFromSrc(
    encoder_channels=(96, 192, 384, 768),
    decoder_channels=256,
    num_classes=3,
    num_groups=32,
)

seg_head_src.eval()


# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

with torch.no_grad():

    logits_src = seg_head_src(
        dummy_features_src,
        output_size=(448, 768),
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_shape = (
    BATCH_SIZE,
    3,
    448,
    768,
)

shape_ok = (
    tuple(logits_src.shape)
    == expected_shape
)

finite_ok = (
    torch.isfinite(logits_src)
    .all()
    .item()
)


print(
    "Output shape:",
    logits_src.shape
)

print(
    "Shape PASS:",
    shape_ok
)

print(
    "Finite PASS:",
    finite_ok
)


assert shape_ok, (
    f"Unexpected output shape: "
    f"{tuple(logits_src.shape)}"
)

assert finite_ok, (
    "NaN or Inf detected in logits."
)


print()
print(
    "Segmentation Head src regression: PASS"
)

Output shape: torch.Size([2, 3, 448, 768])
Shape PASS: True
Finite PASS: True

Segmentation Head src regression: PASS


## 5-2. Segmentation Dataset 설계

### 목적

Rasterization까지 검증된 데이터를 실제 Segmentation 학습 입력으로
연결하기 위한 Dataset 구조를 설계한다.

현재는 구현을 확정하지 않고 설계 contract만 보존한다.

### 이미 확정된 사항

- source annotation: `modelA_segmentation_annotations_v3.jsonl`
- RGB / GT는 동일한 `TransformConfig` 사용
- baseline input: 768×448
- target channels:
  - 0: traffic_lane
  - 1: stop_line
  - 2: crosswalk
- GT는 `rasterizer.py`를 통해 runtime 생성
- mask PNG를 사전에 고정 생성하지 않음

### 향후 결정 / 통합할 사항

1. 최종 Dataset return format
2. Model A 공통 RGB preprocessing 연결
3. augmentation 정책
4. train / validation sample 전달 방식
5. annotation loading / grouping 구조
6. metadata 반환 여부
7. tensor dtype / value range


## 5-3. Dataset 반환 형식

Segmentation Dataset은 tuple보다 dictionary 형태를 baseline 후보로 유지한다.

한 sample의 기본 구조 후보:

- `image`
  - Model A Shared Encoder에 입력될 RGB image tensor

- `seg_target`
  - Segmentation Head 학습용 GT
  - channel 0: traffic_lane
  - channel 1: stop_line
  - channel 2: crosswalk

- `file_name`
  - QC / debugging / prediction 추적을 위한 원본 이미지 파일명

현재 baseline shape은 768×448이지만,
Dataset 내부에서 해당 크기를 hardcoding하지 않고
`TransformConfig → TransformInfo` 결과를 따른다.

향후 Vector Task와 통합할 경우
동일한 sample dictionary에 vector 관련 target을 확장할 수 있도록 한다.


## 5-4. RGB Dataset 출력 및 Normalization

### RGB tensor 후보 규격

Dataset / 공통 preprocessing 이후 RGB는 다음 형태를 사용한다.

- color: RGB
- dtype: `torch.float32`
- shape: `[3, H, W]`

### Shared Encoder에서 확인된 사항

주한의 `ModelAEncoder` Wrapper 내부에서는
Resize / Tensor 변환 / Normalize / Padding을 수행하지 않는다.

즉 ConvNeXt pretrained normalization은
**Shared Encoder 입력 전에 RGB Tensor에 적용되어야 한다.**

최종 Multi-task 통합 시 원칙:

1. RGB / Segmentation GT / Vector point의 **공간 변환**은
   공통 `TransformConfig / TransformInfo`를 기준으로 맞춘다.
2. ConvNeXt mean/std normalization은 **RGB에만 적용한다.**
3. normalization을 Segmentation Dataset 전용 로직으로 중복 hardcoding하지 않는다.

### 통합 시 확인할 부분

현재 주한 Encoder Notebook의 RGB transform과
`src/preprocessing/transforms.py`의 spatial transform 구현은
최종 Multi-task 연결 시 하나의 공통 preprocessing contract로 정리한다.


## 5-5. Augmentation 정책

### Baseline

첫 Segmentation 학습 baseline에서는
추가적인 data augmentation을 적용하지 않는다.

적용되는 geometric transform은 현재 검증 완료된 공통 preprocessing만 사용한다.

- resize
- padding

이유:

1. Dataset / Seg Head / Loss의 정상 동작을 먼저 검증한다.
2. 초기 학습 문제 발생 시 augmentation을 원인에서 제외할 수 있다.
3. Model A는 Segmentation과 Vector가 동일한 RGB/좌표계를 공유하므로
   Segmentation 전용 geometric augmentation을 독립적으로 적용하지 않는다.

### 이후 실험

Baseline 학습이 안정화된 이후
geometry를 변경하지 않는 photometric augmentation부터 비교한다.

예:

- brightness
- contrast
- color variation

Horizontal flip, crop, rotation, affine, perspective 등
geometry를 변경하는 augmentation은
RGB / Seg GT / Vector point가 동일하게 변환되는
공통 augmentation pipeline이 마련된 경우에만 검토한다.


## 5-6. Train / Validation Split 정책

### 기본 원칙

Dataset 내부에서 임의로 train / validation을 분할하지 않는다.

Dataset은 외부에서 결정된 sample 목록을 받아 사용하도록 설계한다.

### Data Leakage 주의

교통 장면 데이터가 영상 또는 연속 프레임 기반일 경우,
같은 scene의 유사 프레임이 Train과 Validation에 동시에 포함되면
Validation 성능이 과대평가될 수 있다.

따라서 가능한 경우 다음 우선순위를 사용한다.

1. video / scene / sequence 단위 Group Split
2. 그룹 정보가 존재하지 않을 경우 deterministic image-level split

### Baseline 후보

- Train: 약 80%
- Validation: 약 20%

단, 실제 비율은 원본 데이터의 scene/group 정보를 확인한 후 확정한다.

Split 결과는 고정하여
실험마다 Validation sample이 변경되지 않도록 한다.


## 5-7. Annotation Loading 정책

### 기본 원칙

v3 JSONL을 `__getitem__()` 호출마다 다시 읽지 않는다.

Dataset 또는 공통 annotation loader 초기화 시
JSONL을 한 번만 parsing하고,
`file_name` 기준으로 annotation을 grouping한다.

예:

```python
{
    "image_a.jpg": [annotation, annotation, ...],
    "image_b.jpg": [annotation, ...],
}
```

학습 중에는 `file_name`을 이용하여
해당 이미지의 annotation만 메모리에서 조회한다.

### 공통화 방향

v3 annotation은 Segmentation뿐 아니라
향후 Vector Task에서도 사용할 정보
(points, lane_type 등)를 포함한다.

따라서 JSONL parsing / grouping 로직은
Segmentation 전용 로직으로 강하게 결합하지 않고
Model A 공통 data loading 기능으로 분리할 수 있도록 설계한다.

정확한 module 위치는 전체 Model A Data Pipeline contract 확정 후 결정한다.

### 재현성

Dataset sample 순서는 deterministic하게 유지한다.

Train / Validation Dataset은
Dataset 내부에서 split하지 않고
외부에서 결정된 `file_name` 목록을 전달받을 수 있도록 한다.


## 5-8. Segmentation Dataset Contract — 현재 보존안

### Input

Segmentation Dataset은 향후 다음 정보를 사용한다.

- 원본 RGB image directory
- v3 annotation 정보
- 외부에서 결정된 `file_name` 목록
- 공통 `TransformConfig`
- Segmentation `RasterConfig`

Dataset 내부에서 Train / Validation split을 수행하지 않는다.

### Output 후보

한 sample은 dictionary 형태로 반환한다.

- `image`
  - RGB
  - `torch.float32`
  - `[3, H, W]`

- `seg_target`
  - `torch.float32`
  - `[3, H, W]`
  - binary value `0.0 / 1.0`
  - channel 0: traffic_lane
  - channel 1: stop_line
  - channel 2: crosswalk

- `file_name`
  - QC / debugging을 위한 원본 이미지 파일명

### Transform

RGB와 Segmentation GT는
동일한 `TransformConfig / TransformInfo`를 사용한다.

현재 baseline:

```text
1920×1080
→ 768×432
→ padding
→ 768×448
```

Dataset 내부에 768×448을 hardcoding하지 않는다.

### Augmentation

초기 baseline에서는 추가 augmentation을 적용하지 않는다.

- resize: 사용
- padding: 사용
- random crop: 사용하지 않음
- horizontal flip: 사용하지 않음
- rotation / affine / perspective: 사용하지 않음
- photometric augmentation: baseline 안정화 이후 실험

### Annotation

v3 JSONL은 매 sample마다 다시 읽지 않는다.

초기화 단계에서 한 번 parsing하고
`file_name` 기준으로 grouping하여 사용한다.

Annotation parsing은 향후 Vector Task와 공유할 수 있도록
Segmentation 전용 구현에 강하게 결합하지 않는다.

### Normalization

ConvNeXt normalization은 Shared Encoder 입력 전에 RGB에만 적용한다.

최종 구현 위치는 주한이 Shared Encoder / Segmentation / Vector를
통합하는 Model A 공통 preprocessing 단계에서 확정한다.


# 6. Loss Design

Head Forward가 정상 동작한 뒤 진행한다.

현재 Segmentation은 3개의 independent binary channel 구조이므로
single-class softmax / class-id 방식으로 자동 확정하지 않는다.

후보:

- `BCEWithLogitsLoss`
- Dice 계열
- BCE + Dice
- class / task weighting

최종 Loss는 실험 후 결정한다.


# 7. Prediction ↔ GT / Training / Validation

Head와 Loss를 검증한 뒤
Dataset / DataLoader를 연결하여 실제 학습 파이프라인으로 확장한다.

진행 예정:

1. Prediction `[B, 3, H, W]`
2. GT `[B, 3, H, W]`
3. Loss 계산
4. Training loop
5. Validation
6. class별 metric
7. Prediction QC
